# 蒙特卡洛算法教程

本教程介绍蒙特卡洛算法的基本原理、应用和实现方法。

## 1. 蒙特卡洛算法简介

蒙特卡洛方法是一类通过随机采样来获得数值结果的计算算法。这种方法在物理、数学、金融等领域有广泛应用。

### 基本原理

蒙特卡洛方法的基本思想是：

1. 构建一个概率模型，使所求问题的解等于该模型中某个随机变量的概率分布或数学期望
2. 对模型进行随机抽样
3. 利用抽样结果计算所求解的近似值

### 数学基础

根据大数定律，如果$X_1, X_2, ..., X_n$是独立同分布的随机变量，且期望$E[X_i] = \mu$存在，则：

$$\bar{X}_n = \frac{1}{n}\sum_{i=1}^{n} X_i \xrightarrow{P} \mu \quad \text{当} \quad n \to \infty$$

其中$\xrightarrow{P}$表示依概率收敛。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns

# Set style for plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 2. 蒙特卡洛积分

蒙特卡洛积分是蒙特卡洛方法最基本的应用之一，用于计算定积分的近似值。

### 基本原理

假设我们要计算积分：

$$I = \int_a^b f(x) dx$$

我们可以将其转化为期望值的形式：

$$I = (b-a) \cdot E[f(X)] \quad \text{其中} \quad X \sim U(a,b)$$

然后通过随机采样来估计这个期望值：

$$I \approx (b-a) \cdot \frac{1}{n}\sum_{i=1}^{n} f(x_i) \quad \text{其中} \quad x_i \sim U(a,b)$$

In [ ]:
def monte_carlo_integration(f, a, b, n=10000):
    """
    Monte Carlo integration of function f from a to b with n samples
    
    Parameters:
    f: function to integrate
    a: lower bound
    b: upper bound
    n: number of samples
    
    Returns:
    Approximate integral value
    """
    # Generate random samples
    x = np.random.uniform(a, b, n)
    
    # Evaluate function at random points
    y = f(x)
    
    # Calculate integral approximation
    integral = (b - a) * np.mean(y)
    
    return integral

In [ ]:
# Example: Calculate integral of sin(x) from 0 to pi
def f(x):
    return np.sin(x)

a, b = 0, np.pi
true_value = 2.0  # True value of integral from 0 to pi of sin(x) dx

# Calculate with different numbers of samples
sample_sizes = [100, 1000, 10000, 100000]
results = []
errors = []

for n in sample_sizes:
    approx = monte_carlo_integration(f, a, b, n)
    error = abs(approx - true_value)
    results.append(approx)
    errors.append(error)
    print(f"Samples: {n:7d}, Approximation: {approx:.6f}, Error: {error:.6f}")

# Plot convergence
plt.figure(figsize=(10, 6))
plt.plot(sample_sizes, errors, 'o-')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Number of Samples')
plt.ylabel('Absolute Error')
plt.title('Monte Carlo Integration Convergence')
plt.grid(True, which="both", ls="--")
plt.show()

## 3. 重要性采样

重要性采样是一种改进蒙特卡洛积分效率的技术，特别适用于函数在某些区域变化剧烈的情况。

### 基本原理

在标准蒙特卡洛积分中，我们使用均匀分布进行采样。但在某些情况下，如果函数在某些区域变化剧烈，均匀采样可能效率低下。

重要性采样使用一个重要性分布$g(x)$来指导采样，使得更多的样本点落在函数值较大的区域。

积分可以重写为：

$$I = \int_a^b \frac{f(x)}{g(x)} g(x) dx = E_g\left[\frac{f(X)}{g(X)}\right]$$

其中$X \sim g(x)$，$g(x)$是重要性分布函数。

### 最优重要性分布

理论上，最优的重要性分布与$|f(x)|$成正比：

$$g_{opt}(x) = \frac{|f(x)|}{\int_a^b |f(x)| dx}$$

但在实际应用中，我们通常选择一个易于采样且与$f(x)$形状相似的分布作为重要性分布。

In [ ]:
def importance_sampling_integration(f, g, g_sampler, a, b, n=10000):
    """
    Importance sampling integration
    
    Parameters:
    f: function to integrate
    g: importance distribution function
    g_sampler: function to sample from importance distribution
    a: lower bound
    b: upper bound
    n: number of samples
    
    Returns:
    Approximate integral value
    """
    # Generate samples from importance distribution
    x = g_sampler(n, a, b)
    
    # Evaluate functions
    f_values = f(x)
    g_values = g(x, a, b)
    
    # Calculate integral approximation
    integral = np.mean(f_values / g_values)
    
    return integral

In [ ]:
# Example: Integrate exp(-x^2) from 0 to 10
def f_exp(x):
    return np.exp(-x**2)

# Exponential distribution as importance distribution
def g_exp(x, a, b):
    # Exponential distribution truncated to [a,b]
    lambda_param = 1.0
    norm = np.exp(-lambda_param * a) - np.exp(-lambda_param * b)
    return lambda_param * np.exp(-lambda_param * x) / norm

def sample_exp(n, a, b):
    # Sample from truncated exponential distribution
    lambda_param = 1.0
    u = np.random.uniform(0, 1, n)
    fa = np.exp(-lambda_param * a)
    fb = np.exp(-lambda_param * b)
    x = -np.log(fa - u * (fa - fb)) / lambda_param
    return x

a, b = 0, 10
true_value = 0.886226925452758  # sqrt(pi)/2

# Compare standard Monte Carlo with importance sampling
sample_sizes = [100, 1000, 10000, 100000]
mc_results = []
is_results = []
mc_errors = []
is_errors = []

for n in sample_sizes:
    # Standard Monte Carlo
    mc_approx = monte_carlo_integration(f_exp, a, b, n)
    mc_error = abs(mc_approx - true_value)
    mc_results.append(mc_approx)
    mc_errors.append(mc_error)
    
    # Importance sampling
    is_approx = importance_sampling_integration(f_exp, g_exp, sample_exp, a, b, n)
    is_error = abs(is_approx - true_value)
    is_results.append(is_approx)
    is_errors.append(is_error)
    
    print(f"Samples: {n:7d}, MC Error: {mc_error:.6f}, IS Error: {is_error:.6f}")

# Plot comparison
plt.figure(figsize=(10, 6))
plt.plot(sample_sizes, mc_errors, 'o-', label='Standard Monte Carlo')
plt.plot(sample_sizes, is_errors, 'o-', label='Importance Sampling')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Number of Samples')
plt.ylabel('Absolute Error')
plt.title('Comparison of Monte Carlo Integration Methods')
plt.legend()
plt.grid(True, which="both", ls="--")
plt.show()

## 4. 马尔可夫链蒙特卡洛（MCMC）

马尔可夫链蒙特卡洛（Markov Chain Monte Carlo, MCMC）是一类用于从复杂概率分布中采样的算法。这些方法在贝叶斯统计、统计物理等领域有广泛应用。

### 基本原理

MCMC方法构建一个马尔可夫链，使其平稳分布为目标分布。通过运行这个马尔可夫链足够长的时间，我们可以得到来自目标分布的样本。

### Metropolis-Hastings算法

Metropolis-Hastings算法是最常用的MCMC方法之一。其基本步骤如下：

1. 从初始状态$x_0$开始
2. 对于$t=1,2,...,N$：
   a. 从提议分布$q(x'|x_t)$中生成候选样本$x'$
   b. 计算接受率：
   $$\alpha = \min\left(1, \frac{p(x')q(x_t|x')}{p(x_t)q(x'|x_t)}\right)$$
   c. 从均匀分布$U(0,1)$中采样$u$
   d. 如果$u \leq \alpha$，则接受$x'$，即$x_{t+1} = x'$；否则拒绝，即$x_{t+1} = x_t$

其中$p(x)$是目标分布，$q(x'|x)$是提议分布。

In [ ]:
def metropolis_hastings(target_log_pdf, proposal_sampler, initial_state, n_samples, burn_in=1000):
    """
    Metropolis-Hastings algorithm
    
    Parameters:
    target_log_pdf: log of target probability density function
    proposal_sampler: function to sample from proposal distribution q(x'|x)
    initial_state: starting point of the chain
    n_samples: number of samples to generate
    burn_in: number of initial samples to discard
    
    Returns:
    Array of samples from target distribution
    """
    # Initialize chain
    current_state = initial_state
    samples = []
    accepted = 0
    
    # Run chain
    for i in range(n_samples + burn_in):
        # Generate proposal
        proposed_state = proposal_sampler(current_state)
        
        # Calculate acceptance probability
        log_alpha = target_log_pdf(proposed_state) - target_log_pdf(current_state)
        
        # Accept or reject
        if np.log(np.random.uniform(0, 1)) < log_alpha:
            current_state = proposed_state
            if i >= burn_in:
                accepted += 1
        
        # Store sample after burn-in
        if i >= burn_in:
            samples.append(current_state)
    
    acceptance_rate = accepted / n_samples
    print(f"Acceptance rate: {acceptance_rate:.4f}")
    
    return np.array(samples)

In [ ]:
# Example: Sample from a bimodal distribution
def bimodal_log_pdf(x):
    """Log of a bimodal distribution: mixture of two normals"""
    return np.log(0.3 * np.exp(-0.5 * ((x - 3) / 1)**2) / np.sqrt(2 * np.pi) +
                  0.7 * np.exp(-0.5 * ((x + 2) / 1.5)**2) / (1.5 * np.sqrt(2 * np.pi)))

def normal_proposal_sampler(current_state, scale=1.0):
    """Sample from normal distribution centered at current state"""
    return current_state + np.random.normal(0, scale)

# Run Metropolis-Hastings
initial_state = 0
n_samples = 10000
burn_in = 1000

samples = metropolis_hastings(
    bimodal_log_pdf,
    lambda x: normal_proposal_sampler(x, scale=2.0),
    initial_state,
    n_samples,
    burn_in
)

# Plot results
plt.figure(figsize=(12, 6))

# Plot trace
plt.subplot(1, 2, 1)
plt.plot(samples[:1000])
plt.xlabel('Iteration')
plt.ylabel('Sample Value')
plt.title('Trace Plot (First 1000 Samples)')

# Plot histogram
plt.subplot(1, 2, 2)
plt.hist(samples, bins=50, density=True, alpha=0.7)

# Plot true distribution
x = np.linspace(-8, 8, 1000)
true_pdf = 0.3 * np.exp(-0.5 * ((x - 3) / 1)**2) / np.sqrt(2 * np.pi) + \
           0.7 * np.exp(-0.5 * ((x + 2) / 1.5)**2) / (1.5 * np.sqrt(2 * np.pi))
plt.plot(x, true_pdf, 'r-', lw=2)
plt.xlabel('x')
plt.ylabel('Density')
plt.title('Sampled Distribution vs True Distribution')

plt.tight_layout()
plt.show()

## 5. 蒙特卡洛在物理中的应用

蒙特卡洛方法在物理学中有广泛应用，特别是在统计物理和量子力学中。

### 伊辛模型模拟

伊辛模型是统计物理中的经典模型，用于描述磁性材料的相变。我们可以使用蒙特卡洛方法（特别是Metropolis算法）来模拟伊辛模型。

伊辛模型的哈密顿量为：

$$H = -J\sum_{\langle i,j \rangle} s_i s_j - h\sum_i s_i$$

其中$s_i = \pm 1$是格点$i$上的自旋，$J$是耦合常数，$h$是外磁场，$\langle i,j \rangle$表示最近邻格点对。

### Metropolis算法步骤

1. 随机选择一个格点$i$
2. 计算翻转该格点自旋的能量变化$\Delta E$
3. 如果$\Delta E \leq 0$，接受翻转
4. 如果$\Delta E > 0$，以概率$e^{-\Delta E/k_B T}$接受翻转
5. 重复以上步骤

In [ ]:
def ising_model_energy(lattice, J=1.0, h=0.0):
    """
    Calculate energy of Ising model
    
    Parameters:
    lattice: 2D array of spins (+1 or -1)
    J: coupling constant
    h: external magnetic field
    
    Returns:
    Total energy
    """
    L, M = lattice.shape
    energy = 0.0
    
    # Sum over nearest neighbors
    for i in range(L):
        for j in range(M):
            # Periodic boundary conditions
            right = (i + 1) % L
            down = (j + 1) % M
            
            # Interaction energy (count each pair once)
            energy -= J * lattice[i, j] * (lattice[right, j] + lattice[i, down])
            
            # External field energy
            energy -= h * lattice[i, j]
    
    return energy

def ising_model_magnetization(lattice):
    """
    Calculate magnetization of Ising model
    
    Parameters:
    lattice: 2D array of spins (+1 or -1)
    
    Returns:
    Magnetization per site
    """
    return np.mean(lattice)

def metropolis_ising_step(lattice, T, J=1.0, h=0.0):
    """
    Perform one Metropolis step for Ising model
    
    Parameters:
    lattice: 2D array of spins (+1 or -1)
    T: temperature
    J: coupling constant
    h: external magnetic field
    
    Returns:
    Updated lattice and acceptance flag
    """
    L, M = lattice.shape
    
    # Random site
    i = np.random.randint(0, L)
    j = np.random.randint(0, M)
    
    # Calculate energy change if we flip this spin
    s = lattice[i, j]
    neighbors_sum = (
        lattice[(i+1)%L, j] + lattice[(i-1)%L, j] +
        lattice[i, (j+1)%M] + lattice[i, (j-1)%M]
    )
    
    dE = 2 * s * (J * neighbors_sum + h)
    
    # Metropolis acceptance criterion
    if dE <= 0 or np.random.random() < np.exp(-dE/T):
        lattice[i, j] = -s  # Flip spin
        return lattice, True
    
    return lattice, False

In [ ]:
# Simulate 2D Ising model
L = 20  # Lattice size
T_min, T_max = 1.0, 4.0  # Temperature range
n_temps = 20
temperatures = np.linspace(T_min, T_max, n_temps)
n_steps = 10000  # Monte Carlo steps
n_measure = 100  # Measurements per temperature

# Arrays to store results
magnetizations = []
energies = []
specific_heats = []
susceptibilities = []

for T in temperatures:
    # Initialize random lattice
    lattice = np.random.choice([-1, 1], size=(L, L))
    
    # Equilibration
    for _ in range(n_steps // 2):
        lattice, _ = metropolis_ising_step(lattice, T)
    
    # Measurement
    mag_samples = []
    energy_samples = []
    
    for _ in range(n_measure):
        # Perform several Monte Carlo steps between measurements
        for _ in range(n_steps // n_measure):
            lattice, _ = metropolis_ising_step(lattice, T)
        
        # Measure observables
        mag = ising_model_magnetization(lattice)
        energy = ising_model_energy(lattice) / (L * L)
        
        mag_samples.append(mag)
        energy_samples.append(energy)
    
    # Calculate averages and fluctuations
    avg_mag = np.mean(np.abs(mag_samples))
    avg_energy = np.mean(energy_samples)
    mag_sq = np.mean([m**2 for m in mag_samples])
    energy_sq = np.mean([e**2 for e in energy_samples])
    
    # Specific heat and susceptibility
    specific_heat = (energy_sq - avg_energy**2) / (T**2)
    susceptibility = (mag_sq - avg_mag**2) / T
    
    magnetizations.append(avg_mag)
    energies.append(avg_energy)
    specific_heats.append(specific_heat)
    susceptibilities.append(susceptibility)
    
    print(f"T = {T:.2f}, Magnetization = {avg_mag:.4f}, Energy = {avg_energy:.4f}")

# Plot results
plt.figure(figsize=(15, 10))

# Plot magnetization
plt.subplot(2, 2, 1)
plt.plot(temperatures, magnetizations, 'o-')
plt.xlabel('Temperature')
plt.ylabel('Magnetization')
plt.title('Magnetization vs Temperature')
plt.grid(True)

# Plot energy
plt.subplot(2, 2, 2)
plt.plot(temperatures, energies, 'o-')
plt.xlabel('Temperature')
plt.ylabel('Energy')
plt.title('Energy vs Temperature')
plt.grid(True)

# Plot specific heat
plt.subplot(2, 2, 3)
plt.plot(temperatures, specific_heats, 'o-')
plt.xlabel('Temperature')
plt.ylabel('Specific Heat')
plt.title('Specific Heat vs Temperature')
plt.grid(True)

# Plot susceptibility
plt.subplot(2, 2, 4)
plt.plot(temperatures, susceptibilities, 'o-')
plt.xlabel('Temperature')
plt.ylabel('Susceptibility')
plt.title('Susceptibility vs Temperature')
plt.grid(True)

plt.tight_layout()
plt.show()

## 6. 蒙特卡洛在金融中的应用

蒙特卡洛方法在金融领域有广泛应用，特别是在期权定价和风险评估中。

### 期权定价

期权是一种金融衍生品，其价值依赖于标的资产（如股票）的价格变动。欧式看涨期权的收益为：

$$C = \max(S_T - K, 0)$$

其中$S_T$是到期日的标的资产价格，$K$是行权价。

在风险中性测度下，期权的当前价格为：

$$C_0 = e^{-rT} E[\max(S_T - K, 0)]$$

其中$r$是无风险利率，$T$是到期时间。

### 几何布朗运动模型

在Black-Scholes模型中，标的资产价格遵循几何布朗运动：

$$dS = \mu S dt + \sigma S dW$$

其中$\mu$是漂移率，$\sigma$是波动率，$dW$是维纳过程的增量。

这个随机微分方程的解为：

$$S_T = S_0 \exp\left(\left(\mu - \frac{\sigma^2}{2}\right)T + \sigma \sqrt{T} Z\right)$$

其中$Z \sim N(0,1)$是标准正态随机变量。

In [ ]:
def generate_gbm_paths(S0, mu, sigma, T, n_steps, n_paths):
    """
    Generate paths of Geometric Brownian Motion
    
    Parameters:
    S0: initial stock price
    mu: drift rate
    sigma: volatility
    T: time horizon
    n_steps: number of time steps
    n_paths: number of paths
    
    Returns:
    Array of shape (n_steps+1, n_paths) with price paths
    """
    dt = T / n_steps
    paths = np.zeros((n_steps + 1, n_paths))
    paths[0] = S0
    
    for t in range(1, n_steps + 1):
        z = np.random.standard_normal(n_paths)
        paths[t] = paths[t-1] * np.exp((mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * z)
    
    return paths

def european_call_option_mc(S0, K, T, r, sigma, n_paths=10000):
    """
    Price European call option using Monte Carlo
    
    Parameters:
    S0: initial stock price
    K: strike price
    T: time to maturity
    r: risk-free rate
    sigma: volatility
    n_paths: number of Monte Carlo paths
    
    Returns:
    Option price and standard error
    """
    # Generate terminal stock prices under risk-neutral measure
    z = np.random.standard_normal(n_paths)
    ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * z)
    
    # Calculate payoffs
    payoffs = np.maximum(ST - K, 0)
    
    # Discount payoffs
    discounted_payoffs = np.exp(-r * T) * payoffs
    
    # Calculate price and standard error
    price = np.mean(discounted_payoffs)
    std_error = np.std(discounted_payoffs) / np.sqrt(n_paths)
    
    return price, std_error

def black_scholes_call(S0, K, T, r, sigma):
    """
    Calculate European call option price using Black-Scholes formula
    
    Parameters:
    S0: initial stock price
    K: strike price
    T: time to maturity
    r: risk-free rate
    sigma: volatility
    
    Returns:
    Option price
    """
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    price = S0 * stats.norm.cdf(d1) - K * np.exp(-r * T) * stats.norm.cdf(d2)
    
    return price

In [ ]:
# Example: Option pricing
S0 = 100.0  # Initial stock price
K = 105.0   # Strike price
T = 1.0     # Time to maturity (in years)
r = 0.05    # Risk-free rate
sigma = 0.2 # Volatility

# Calculate option price using Monte Carlo with different numbers of paths
n_paths_list = [1000, 5000, 10000, 50000, 100000]
mc_prices = []
std_errors = []

for n_paths in n_paths_list:
    price, std_error = european_call_option_mc(S0, K, T, r, sigma, n_paths)
    mc_prices.append(price)
    std_errors.append(std_error)
    print(f"Paths: {n_paths:6d}, MC Price: {price:.6f}, Std Error: {std_error:.6f}")

# Calculate exact price using Black-Scholes formula
bs_price = black_scholes_call(S0, K, T, r, sigma)
print(f"\nBlack-Scholes Price: {bs_price:.6f}")

# Plot convergence
plt.figure(figsize=(10, 6))
plt.errorbar(n_paths_list, mc_prices, yerr=1.96*np.array(std_errors), fmt='o-', label='Monte Carlo')
plt.axhline(y=bs_price, color='r', linestyle='--', label='Black-Scholes')
plt.xscale('log')
plt.xlabel('Number of Paths')
plt.ylabel('Option Price')
plt.title('Monte Carlo Option Pricing Convergence')
plt.legend()
plt.grid(True)
plt.show()

# Generate sample price paths
n_steps = 252  # Trading days in a year
n_paths = 10
paths = generate_gbm_paths(S0, r, sigma, T, n_steps, n_paths)

# Plot sample paths
plt.figure(figsize=(10, 6))
time = np.linspace(0, T, n_steps + 1)
for i in range(n_paths):
    plt.plot(time, paths[:, i], lw=1.5)
plt.axhline(y=K, color='r', linestyle='--', label='Strike Price')
plt.xlabel('Time (Years)')
plt.ylabel('Stock Price')
plt.title('Sample Stock Price Paths')
plt.legend()
plt.grid(True)
plt.show()

## 7. 蒙特卡洛方法的高级主题

### 方差缩减技术

方差缩减技术是提高蒙特卡洛模拟效率的重要方法。常见的方差缩减技术包括：

1. **重要性采样**：我们已经在前面的章节中介绍过，通过选择合适的采样分布来减少方差。

2. **控制变量法**：利用与所估计量相关的另一个随机变量（控制变量）来减少方差。

3. **对偶变量法**：利用负相关的随机变量来减少方差。

4. **分层抽样**：将样本空间划分为若干层，从每层中独立抽样。

### 准蒙特卡洛方法

准蒙特卡洛方法使用低差异序列（如Halton序列、Sobol序列）代替伪随机数，可以提供更均匀的样本空间覆盖，从而加速收敛。

准蒙特卡洛方法的误差通常为$O((\log N)^d/N)$，而标准蒙特卡洛方法的误差为$O(1/\sqrt{N})$，其中$d$是问题的维数，$N$是样本数。

In [ ]:
def halton_sequence(n, d):
    """
    Generate Halton sequence
    
    Parameters:
    n: number of points
    d: dimension
    
    Returns:
    Array of shape (n, d) with Halton sequence points
    """
    def halton_single(n, base):
        """Generate n-th point in 1D Halton sequence with given base"""
        result = 0.0
        f = 1.0 / base
        i = n
        while i > 0:
            result += f * (i % base)
            i = i // base
            f = f / base
        return result
    
    # First d primes as bases
    bases = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47][:d]
    
    sequence = np.zeros((n, d))
    for i in range(n):
        for j in range(d):
            sequence[i, j] = halton_single(i + 1, bases[j])
    
    return sequence

def quasi_monte_carlo_integration(f, a, b, n=10000):
    """
    Quasi-Monte Carlo integration using Halton sequence
    
    Parameters:
    f: function to integrate
    a: lower bound
    b: upper bound
    n: number of samples
    
    Returns:
    Approximate integral value
    """
    # Generate Halton sequence in [0,1]^d
    halton_points = halton_sequence(n, 1)
    
    # Scale to [a,b]
    x = a + (b - a) * halton_points.flatten()
    
    # Evaluate function at points
    y = f(x)
    
    # Calculate integral approximation
    integral = (b - a) * np.mean(y)
    
    return integral

In [ ]:
# Compare standard Monte Carlo with Quasi-Monte Carlo
def test_function(x):
    """Test function for integration"""
    return np.sin(x) * np.exp(-x)

a, b = 0, 5
true_value = 0.5 * (1 - np.exp(-5) * (np.sin(5) + np.cos(5)))  # Analytical solution

# Calculate with different numbers of samples
sample_sizes = [100, 1000, 10000]
mc_results = []
qmc_results = []
mc_errors = []
qmc_errors = []

for n in sample_sizes:
    # Standard Monte Carlo
    mc_approx = monte_carlo_integration(test_function, a, b, n)
    mc_error = abs(mc_approx - true_value)
    mc_results.append(mc_approx)
    mc_errors.append(mc_error)
    
    # Quasi-Monte Carlo
    qmc_approx = quasi_monte_carlo_integration(test_function, a, b, n)
    qmc_error = abs(qmc_approx - true_value)
    qmc_results.append(qmc_approx)
    qmc_errors.append(qmc_error)
    
    print(f"Samples: {n:5d}, MC Error: {mc_error:.6f}, QMC Error: {qmc_error:.6f}")

# Visualize sampling patterns
plt.figure(figsize=(15, 5))

# Plot random sampling
plt.subplot(1, 3, 1)
random_samples = np.random.uniform(0, 1, 1000)
plt.hist(random_samples, bins=30, density=True, alpha=0.7)
plt.title('Random Sampling')
plt.xlabel('x')
plt.ylabel('Density')

# Plot Halton sequence
plt.subplot(1, 3, 2)
halton_samples = halton_sequence(1000, 1).flatten()
plt.hist(halton_samples, bins=30, density=True, alpha=0.7)
plt.title('Halton Sequence')
plt.xlabel('x')
plt.ylabel('Density')

# Plot comparison
plt.subplot(1, 3, 3)
plt.plot(sample_sizes, mc_errors, 'o-', label='Standard Monte Carlo')
plt.plot(sample_sizes, qmc_errors, 'o-', label='Quasi-Monte Carlo')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Number of Samples')
plt.ylabel('Absolute Error')
plt.title('Convergence Comparison')
plt.legend()
plt.grid(True, which="both", ls="--")

plt.tight_layout()
plt.show()

## 8. 总结

本教程介绍了蒙特卡洛方法的基本原理和几个重要应用领域：

1. **蒙特卡洛积分**：通过随机采样计算定积分的近似值
2. **重要性采样**：通过选择合适的采样分布提高蒙特卡洛积分的效率
3. **马尔可夫链蒙特卡洛（MCMC）**：用于从复杂概率分布中采样的方法
4. **物理系统模拟**：以伊辛模型为例，展示蒙特卡洛在统计物理中的应用
5. **金融应用**：使用蒙特卡洛方法进行期权定价
6. **高级主题**：介绍方差缩减技术和准蒙特卡洛方法

蒙特卡洛方法的核心思想是通过随机采样来解决确定性问题。虽然这种方法在概念上很简单，但它在许多复杂问题中提供了强大的数值工具，特别是在高维问题中，传统的数值方法可能失效。

随着计算能力的提高，蒙特卡洛方法在科学计算、工程、金融等领域的应用将继续扩大。同时，新的算法和技术也将不断涌现，进一步提高蒙特卡洛方法的效率和适用范围。